In [22]:
import sys
from pathlib import Path
import rootutils

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
project_root = rootutils.setup_root(project_root, indicator=".project-root", pythonpath=True)

## Step 1: Import Libraries and Setup

Import all necessary libraries for training and analysis.

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import pickle

from src.data.components.transform_subset import TransformSubset
from src.data.components.transforms import MediumTransforms
from src.models.components.mobile_net import MobileNet

print("✓ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Libraries imported successfully
PyTorch version: 2.8.0
CUDA available: True


## Step 2: Define Training Functions

Create helper functions for training, validation, and prediction.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{total_loss/(pbar.n+1):.3f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return total_loss / len(dataloader), 100. * correct / total


def get_predictions(model, dataloader, device):
    """Get predictions and labels from a dataloader."""
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Getting predictions", leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())
    
    return np.vstack(all_probs), np.concatenate(all_labels)

print("✓ Training functions defined")

✓ Training functions defined


## Step 3: Configure Training Parameters

Set hyperparameters for the cross-validation training.

In [35]:
# Configuration
data_dir = project_root / "data" / "surface"
train_path = data_dir / "train"
save_dir = project_root / "logs" / "cv_cleanlab"
save_dir.mkdir(parents=True, exist_ok=True)

# Hyperparameters
n_folds = 3
num_epochs = 10
batch_size = 32
lr = 0.01
seed = 42
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Training Configuration:")
print(f"  Data path: {train_path}")
print(f"  Save dir: {save_dir}")
print(f"  Number of folds: {n_folds}")
print(f"  Epochs per fold: {num_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {lr}")
print(f"  Device: {device}")

Training Configuration:
  Data path: /home/lukasb/Documents/NoisyLabelDefectDetection/data/surface/train
  Save dir: /home/lukasb/Documents/NoisyLabelDefectDetection/logs/cv_cleanlab
  Number of folds: 3
  Epochs per fold: 10
  Batch size: 32
  Learning rate: 0.01
  Device: cuda


## Step 4: Load Dataset and Setup Cross-Validation

Load the full training dataset and prepare k-fold splits.

In [36]:
# Load full dataset
full_dataset = ImageFolder(str(train_path))

num_samples = len(full_dataset)
num_classes = len(full_dataset.classes)

print(f"Dataset loaded:")
print(f"  Total samples: {num_samples}")
print(f"  Number of classes: {num_classes}")
print(f"  Classes: {full_dataset.classes}")

# Setup k-fold cross-validation
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
print(f"\n✓ {n_folds}-fold cross-validation setup complete")

Dataset loaded:
  Total samples: 3759
  Number of classes: 10
  Classes: ['bend', 'black_stain', 'corrosion', 'crack', 'deformation', 'missing_part', 'no_deficiencies', 'other', 'silicate_stain', 'water_stain']

✓ 3-fold cross-validation setup complete


## Step 5: Train All Folds

Train MobileNetV3-Large on each fold. This will take 30-60 minutes depending on your hardware.

In [37]:
# Storage for results
all_indices = []
all_probs = []
all_labels = []
fold_results = []

# Setup transforms
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
train_transforms = MediumTransforms.train_transforms(mean, std)
eval_transforms = MediumTransforms.eval_transforms(mean, std)

# Train each fold
for fold_idx, (train_indices, val_indices) in enumerate(kfold.split(range(num_samples))):
    print(f"\n{'='*60}")
    print(f"Training Fold {fold_idx + 1}/{n_folds}")
    print(f"{'='*60}")
    
    # Set seed for reproducibility
    torch.manual_seed(seed + fold_idx)
    np.random.seed(seed + fold_idx)
    
    # Create fold datasets
    train_dataset = TransformSubset(
        full_dataset, train_indices.tolist(), transform=train_transforms, return_index=False
    )
    val_dataset = TransformSubset(
        full_dataset, val_indices.tolist(), transform=eval_transforms, return_index=False
    )
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True
    )
    
    # Create model
    model = MobileNet(num_classes=num_classes, pretrained=True, variant='large')
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    # Training loop
    best_val_acc = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}", end=" - ")
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # Update learning rate
        scheduler.step()
        
        print(f"Train: {train_loss:.3f}/{train_acc:.1f}% | Val: {val_loss:.3f}/{val_acc:.1f}%", end="")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            print(" ✓ (best)", end="")
        print()
    
    print(f"\n✓ Fold {fold_idx + 1} complete - Best Val Acc: {best_val_acc:.2f}%")
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    # Save best model
    fold_save_dir = save_dir / f"fold_{fold_idx}"
    fold_save_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = fold_save_dir / "best_model.pt"
    torch.save({
        'model_state_dict': best_model_state,
        'val_acc': best_val_acc,
        'num_classes': num_classes,
    }, best_model_path)
    
    # Get predictions on validation set (out-of-fold predictions)
    val_probs, val_labels = get_predictions(model, val_loader, device)
    
    # Store results
    fold_results.append({
        "fold_idx": fold_idx,
        "val_indices": val_indices.tolist(),
        "best_val_acc": best_val_acc,
        "best_model_path": str(best_model_path),
    })
    all_indices.extend(val_indices.tolist())
    all_probs.append(val_probs)
    all_labels.extend(val_labels)

print(f"\n{'='*60}")
print("ALL FOLDS COMPLETE")
print(f"{'='*60}")
avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
print(f"Average validation accuracy: {avg_acc:.2f}%")
for result in fold_results:
    print(f"  Fold {result['fold_idx'] + 1}: {result['best_val_acc']:.2f}%")


Training Fold 1/3
Train samples: 2506, Val samples: 1253
Epoch 1/10 - 

Train: 1.648/43.1% | Val: 1.819/37.5% ✓ (best)
Epoch 2/10 - 

Train: 1.174/57.8% | Val: 1.281/55.5% ✓ (best)
Epoch 3/10 - 

Train: 0.881/69.0% | Val: 1.165/60.6% ✓ (best)
Epoch 4/10 - 

Train: 0.670/76.1% | Val: 1.483/52.5%
Epoch 5/10 - 

...

Train: 0.277/90.5% | Val: 0.712/78.2% ✓ (best)
Epoch 8/10 - 

Train: 0.240/91.4% | Val: 0.602/81.1% ✓ (best)
Epoch 9/10 - 

Train: 0.185/93.7% | Val: 0.566/82.5% ✓ (best)
Epoch 10/10 - 

Train: 0.188/93.7% | Val: 0.558/83.2% ✓ (best)

✓ Fold 3 complete - Best Val Acc: 83.16%



ALL FOLDS COMPLETE
Average validation accuracy: 83.16%
  Fold 1: 82.60%
  Fold 2: 83.72%
  Fold 3: 83.16%


## Step 6: Combine and Save Predictions

Combine out-of-fold predictions from all folds and save to disk.

In [38]:
# Combine all out-of-fold predictions
all_probs = np.vstack(all_probs)
all_labels = np.array(all_labels)
all_indices = np.array(all_indices)

# Sort by original indices to align with dataset
sort_idx = np.argsort(all_indices)
pred_probs = all_probs[sort_idx]
labels = all_labels[sort_idx]
all_indices = all_indices[sort_idx]

# Save results
cv_results = {
    "pred_probs": pred_probs,
    "labels": labels,
    "indices": all_indices,
    "classes": full_dataset.classes,
    "class_to_idx": full_dataset.class_to_idx,
    "samples": full_dataset.samples,
    "n_folds": n_folds,
    "fold_results": fold_results,
}

output_path = save_dir / "cv_predictions.pkl"
with open(output_path, "wb") as f:
    pickle.dump(cv_results, f)

print(f"✓ Saved predictions to: {output_path}")
print(f"  Prediction shape: {pred_probs.shape}")
print(f"  Labels shape: {labels.shape}")
print(f"\nReady for cleanlab analysis!")

✓ Saved predictions to: /home/lukasb/Documents/NoisyLabelDefectDetection/logs/cv_cleanlab/cv_predictions.pkl
  Prediction shape: (3759, 10)
  Labels shape: (3759,)

Ready for cleanlab analysis!


## Step 7: Load Predictions (if already trained)

If you've already completed the training, you can skip to here and load the saved predictions.

In [39]:
# Only run this cell if you want to reload predictions from disk
# (Skip if you just ran the training above)

cv_results_path = project_root / "logs" / "cv_cleanlab" / "cv_predictions.pkl"

if cv_results_path.exists():
    with open(cv_results_path, "rb") as f:
        cv_results = pickle.load(f)
    
    pred_probs = cv_results["pred_probs"]
    labels = cv_results["labels"]
    classes = cv_results["classes"]
    samples = cv_results["samples"]
    
    print("✓ Loaded cross-validation predictions from disk")
    print(f"  Samples: {len(labels)}")
    print(f"  Classes: {len(classes)} - {classes}")
    print(f"  Prediction shape: {pred_probs.shape}")
    print(f"  Folds: {cv_results['n_folds']}")
else:
    print(f"❌ Predictions file not found at: {cv_results_path}")
    print("Please run the training cells above first")

✓ Loaded cross-validation predictions from disk
  Samples: 3759
  Classes: 10 - ['bend', 'black_stain', 'corrosion', 'crack', 'deformation', 'missing_part', 'no_deficiencies', 'other', 'silicate_stain', 'water_stain']
  Prediction shape: (3759, 10)
  Folds: 3


## Step 8: Install and Import Cleanlab

Install cleanlab if not already installed.

In [45]:
# Install cleanlab
# !pip install cleanlab

from cleanlab.filter import find_label_issues
from cleanlab.rank import get_label_quality_scores
from cleanlab import Datalab
import pandas as pd

print("✓ Cleanlab imported successfully")

✓ Cleanlab imported successfully


## Step 9: Find Label Issues with Cleanlab

Use cleanlab to identify potential label errors.

In [47]:
ds = {"image": [s[0] for s in samples], "label": labels}
lab = Datalab(data=ds, label_name="label", image_key="image")

AssertionError: 

## Step 10: Analyze Label Issues

Create a detailed analysis of the identified label issues.

In [44]:
# Create a DataFrame with detailed information
results_df = pd.DataFrame({
    'sample_idx': range(len(labels)),
    'image_path': [sample[0] for sample in samples],
    'given_label': labels,
    'given_class_name': [classes[label] for label in labels],
    'predicted_label': pred_probs.argmax(axis=1),
    'predicted_class_name': [classes[pred] for pred in pred_probs.argmax(axis=1)],
    'label_quality_score': label_quality_scores,
    'is_label_issue': label_issues_mask,
    'confidence_in_given_label': [pred_probs[i, labels[i]] for i in range(len(labels))],
    'max_predicted_prob': pred_probs.max(axis=1),
})

# Sort by label quality score (lowest first = most problematic)
results_df = results_df.sort_values('label_quality_score')

print("✓ Created detailed analysis DataFrame")
print(f"\nColumns: {list(results_df.columns)}")
print(f"\nTop 10 most problematic samples:")
print(results_df.head(10)[['given_class_name', 'predicted_class_name', 
                             'label_quality_score', 'confidence_in_given_label']])

ValueError: All arrays must be of the same length

## Step 11: Visualize Label Issues

Visualize the distribution of label issues across classes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Distribution of label quality scores
ax1 = axes[0, 0]
ax1.hist(label_quality_scores, bins=50, edgecolor='black', alpha=0.7)
ax1.axvline(label_quality_scores[label_issues_mask].max(), 
            color='red', linestyle='--', label='Issue threshold')
ax1.set_xlabel('Label Quality Score')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Label Quality Scores')
ax1.legend()

# 2. Issues per class
ax2 = axes[0, 1]
issues_per_class = results_df[results_df['is_label_issue']].groupby('given_class_name').size()
total_per_class = results_df.groupby('given_class_name').size()
issue_rate = (issues_per_class / total_per_class * 100).fillna(0).sort_values(ascending=False)
issue_rate.plot(kind='bar', ax=ax2, color='coral')
ax2.set_xlabel('Class')
ax2.set_ylabel('Label Issue Rate (%)')
ax2.set_title('Label Issue Rate by Class')
ax2.tick_params(axis='x', rotation=45)

# 3. Confusion matrix of given vs predicted labels (for issues only)
ax3 = axes[1, 0]
issues_df = results_df[results_df['is_label_issue']]
confusion_data = pd.crosstab(
    issues_df['given_class_name'], 
    issues_df['predicted_class_name']
)
sns.heatmap(confusion_data, annot=True, fmt='d', cmap='YlOrRd', ax=ax3, cbar_kws={'label': 'Count'})
ax3.set_xlabel('Predicted Class')
ax3.set_ylabel('Given Class')
ax3.set_title('Confusion Matrix (Label Issues Only)')
ax3.tick_params(axis='x', rotation=45)
ax3.tick_params(axis='y', rotation=45)

# 4. Label quality score vs confidence
ax4 = axes[1, 1]
colors = ['red' if issue else 'blue' for issue in label_issues_mask]
ax4.scatter(label_quality_scores, results_df['confidence_in_given_label'], 
           c=colors, alpha=0.3, s=10)
ax4.set_xlabel('Label Quality Score')
ax4.set_ylabel('Confidence in Given Label')
ax4.set_title('Label Quality vs Model Confidence')
ax4.legend(['Label Issues', 'Clean Labels'], loc='lower right')

plt.tight_layout()
plt.show()

print(f"\n✓ Visualizations created")

## Step 12: Display Most Problematic Images

Visualize the actual images with the lowest label quality scores.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Get top N most problematic samples
n_samples = 20
top_issues = results_df[results_df['is_label_issue']].head(n_samples)

# Create figure
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.ravel()

for idx, (_, row) in enumerate(top_issues.iterrows()):
    if idx >= n_samples:
        break
    
    # Load and display image
    img_path = Path(row['image_path'])
    if img_path.exists():
        img = Image.open(img_path)
        axes[idx].imshow(img)
    else:
        axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
    
    # Add title with information
    title = f"Given: {row['given_class_name']}\n"
    title += f"Predicted: {row['predicted_class_name']}\n"
    title += f"Quality: {row['label_quality_score']:.3f}"
    axes[idx].set_title(title, fontsize=9)
    axes[idx].axis('off')

plt.suptitle('Top 20 Most Problematic Samples', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

print(f"✓ Displayed {min(n_samples, len(top_issues))} most problematic images")

## Step 13: Export Results

Save the analysis results for further review or data cleaning.

In [ ]:
# Save full results to CSV
output_dir = project_root / "logs" / "cleanlab_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

# Save all samples
all_samples_path = output_dir / "all_samples_analysis.csv"
results_df.to_csv(all_samples_path, index=False)
print(f"✓ Saved all samples to: {all_samples_path}")

# Save only label issues
issues_path = output_dir / "label_issues.csv"
results_df[results_df['is_label_issue']].to_csv(issues_path, index=False)
print(f"✓ Saved label issues to: {issues_path}")

# Save summary statistics
summary = {
    'total_samples': len(labels),
    'num_issues': num_issues,
    'issue_percentage': 100 * num_issues / len(labels),
    'num_classes': len(classes),
    'mean_quality_score': float(label_quality_scores.mean()),
    'std_quality_score': float(label_quality_scores.std()),
}

summary_path = output_dir / "summary_stats.txt"
with open(summary_path, 'w') as f:
    for key, value in summary.items():
        f.write(f"{key}: {value}\n")
print(f"✓ Saved summary to: {summary_path}")

print(f"\n{'='*60}")
print("CLEANLAB ANALYSIS COMPLETE")
print(f"{'='*60}")
print(f"Total samples analyzed: {summary['total_samples']}")
print(f"Label issues found: {summary['num_issues']} ({summary['issue_percentage']:.1f}%)")
print(f"Results saved to: {output_dir}")
print(f"{'='*60}")